# Transfer Learning en Keras con Cats vs Dogs

En este notebook construiremos un clasificador de imágenes usando **transfer learning**.

La idea será:

1. cargar un dataset de imágenes de gatos y perros,
2. preparar los conjuntos de entrenamiento, validación y test,
3. usar un modelo **preentrenado**,
4. congelar inicialmente la base del modelo,
5. entrenar una nueva cabeza de clasificación,
6. y luego probar un pequeño **fine-tuning**.

En este notebook usaremos **MobileNetV2** como modelo base.

## ¿Qué queremos aprender en este notebook?

Al finalizar este notebook deberías poder:

- entender la lógica general de **transfer learning**,
- usar un modelo preentrenado desde `keras.applications`,
- distinguir entre **feature extraction** y **fine-tuning**,
- construir un modelo con la **Functional API**,
- y evaluar si reutilizar un modelo preentrenado mejora el desempeño en una tarea de visión.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print("TensorFlow version:", tf.__version__)

## Dataset a utilizar

Usaremos el clásico dataset **Cats vs Dogs**.

Es un problema binario de clasificación de imágenes:

- clase 0: gato
- clase 1: perro

Más adelante separaremos los datos en:

- entrenamiento,
- validación,
- test.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Ruta al dataset en Google Drive

Ahora usaremos el dataset almacenado en Google Drive.

La carpeta del dataset ya viene organizada en tres subconjuntos:

- `train`
- `validation`
- `test`

y dentro de cada uno esperamos encontrar dos subcarpetas:

- `cats`
- `dogs`

In [ ]:
import os

# Ajusta esta ruta según tu Google Drive
base_dir = "/content/drive/MyDrive/data/cats-vs-dogs_small"

train_dir = os.path.join(base_dir, "train")
validation_dir = os.path.join(base_dir, "validation")
test_dir = os.path.join(base_dir, "test")

print("Base dir:", base_dir)
print("Train dir:", train_dir)
print("Validation dir:", validation_dir)
print("Test dir:", test_dir)

## Verificación de rutas

Antes de seguir, conviene verificar que existan:

- la carpeta base,
- `train`,
- `validation`,
- `test`.

In [ ]:
print("¿Existe base_dir?:", os.path.exists(base_dir))
print("¿Existe train_dir?:", os.path.exists(train_dir))
print("¿Existe validation_dir?:", os.path.exists(validation_dir))
print("¿Existe test_dir?:", os.path.exists(test_dir))

## Inspección rápida de la estructura

Ahora revisaremos el contenido de las carpetas para confirmar que el dataset está organizado como esperamos.

In [ ]:
print("Contenido de base_dir:", os.listdir(base_dir))
print("Contenido de train_dir:", os.listdir(train_dir))
print("Contenido de validation_dir:", os.listdir(validation_dir))
print("Contenido de test_dir:", os.listdir(test_dir))

In [6]:
img_size = (160, 160)
batch_size = 32
seed = 123

## Observación sobre este dataset

En este caso, el dataset ya viene separado en:

- `train`
- `validation`
- `test`

Por lo tanto, no necesitamos crear particiones adicionales.
Usaremos cada carpeta directamente para su propósito correspondiente.

## Carga de los datasets

Usaremos `image_dataset_from_directory`, que ya hemos utilizado en sesiones anteriores.

Esto nos permite:

- leer imágenes desde carpetas,
- asignar etiquetas automáticamente según los nombres de subcarpetas,
- crear batches,
- y trabajar directamente con `tf.data.Dataset`.

In [ ]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    label_mode="binary",
    image_size=img_size,
    batch_size=batch_size,
    shuffle=True,
    seed=seed
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    validation_dir,
    label_mode="binary",
    image_size=img_size,
    batch_size=batch_size,
    shuffle=False
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    test_dir,
    label_mode="binary",
    image_size=img_size,
    batch_size=batch_size,
    shuffle=False
)

## Nombres de clases

Veamos cómo quedaron codificadas las clases.

In [ ]:
class_names = train_ds.class_names
print("Class names:", class_names)

## Visualización de algunas imágenes

Antes de entrenar cualquier modelo, siempre conviene mirar ejemplos del dataset.

Eso ayuda a verificar:

- que las imágenes se estén cargando correctamente,
- que el tamaño sea razonable,
- y que las etiquetas tengan sentido.

In [ ]:
plt.figure(figsize=(10, 10))

for images, labels in train_ds.take(1):
    for i in range(9):
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        plt.title(class_names[int(labels[i].numpy()[0])])
        plt.axis("off")

plt.show()

## Optimización del pipeline

Usaremos `prefetch` para mejorar el rendimiento del pipeline de datos.

La idea es que mientras el modelo está entrenando un batch, TensorFlow puede ir preparando el siguiente.

In [11]:
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)
test_ds = test_ds.prefetch(buffer_size=AUTOTUNE)

## Data augmentation

Como nuestro dataset no es gigantesco, conviene aplicar una pequeña estrategia de **data augmentation**.

Esto ayuda a que el modelo vea variaciones de las imágenes durante el entrenamiento, por ejemplo:

- flips horizontales,
- pequeñas rotaciones,
- pequeños zooms.

Importante:
el [augmentation](https://keras.io/api/layers/preprocessing_layers/image_augmentation/) se aplica solo durante entrenamiento.

In [12]:
data_augmentation = keras.Sequential(
    [
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.1),
        layers.RandomZoom(0.1),
    ],
    name="data_augmentation"
)

## Visualizar el efecto del augmentation

Esto no cambia el dataset original guardado en disco.

Simplemente genera versiones modificadas "al vuelo" durante el entrenamiento.

In [ ]:
for images, _ in train_ds.take(1):
    sample_image = images[0]

plt.figure(figsize=(10, 10))
for i in range(9):
    augmented_image = data_augmentation(tf.expand_dims(sample_image, 0))
    ax = plt.subplot(3, 3, i + 1)
    plt.imshow(augmented_image[0].numpy().astype("uint8"))
    plt.axis("off")

plt.show()

## ¿Qué es [MobileNetV2](https://arxiv.org/pdf/1801.04381)?

Ahora pasamos al corazón del notebook.

**MobileNetV2** es una arquitectura de redes convolucionales diseñada para ser:

- más liviana,
- más eficiente,
- y más rápida que otras arquitecturas más pesadas.

Es especialmente útil cuando queremos:

- entrenar o inferir con menos costo computacional,
- trabajar en dispositivos con recursos limitados,
- o usar transfer learning de forma práctica en entornos como Colab.

### Idea general
MobileNetV2 sigue siendo una CNN, pero está diseñada de manera más eficiente que una CNN tradicional.

Dos ideas importantes son:

- el uso de **depthwise separable convolutions**, que reducen el costo computacional,
- y los llamados **inverted residual blocks**, que permiten representar información de manera eficiente.

### En este curso
No necesitamos entrar en todos los detalles matemáticos internos.

Lo importante aquí es entender que:

- es un modelo moderno y eficiente,
- viene disponible preentrenado en `keras.applications`,
- y podemos reutilizarlo como base para nuestra nueva tarea.

## Una idea clave: depthwise separable convolution

Una de las razones por las que MobileNetV2 es eficiente es que no usa siempre convoluciones estándar del modo tradicional.

En su lugar, usa una idea llamada **depthwise separable convolution**.

### Recordatorio
En una convolución estándar, cada filtro mira simultáneamente:

- la dimensión espacial de la imagen,
- y todos los canales de entrada.

Eso funciona bien, pero puede ser costoso en cantidad de operaciones y parámetros.

### Idea general
La **depthwise separable convolution** divide ese proceso en dos pasos:

1. **Depthwise convolution**  
   aplica un filtro por cada canal de entrada, por separado.

2. **Pointwise convolution**  
   combina la información entre canales usando convoluciones `1x1`.

La idea es separar:

- el aprendizaje espacial,
- del aprendizaje entre canales.

## Intuición de por qué esto ayuda

Esta descomposición suele reducir bastante el costo computacional comparado con una convolución estándar.

En términos intuitivos:

- primero detectamos patrones espaciales dentro de cada canal,
- luego mezclamos la información entre canales.

Eso permite construir redes más livianas sin perder completamente la capacidad de aprender representaciones útiles.

### Mensaje práctico
No necesitamos calcular aquí toda la aritmética de parámetros y multiplicaciones.

Lo importante es entender que esta estrategia hace que MobileNetV2 sea mucho más eficiente que una CNN tradicional de tamaño comparable.

## Otra idea importante en MobileNetV2: inverted residual blocks

Además de usar convoluciones eficientes, MobileNetV2 organiza sus capas en bloques llamados **inverted residual blocks**.

La intuición es la siguiente:

- primero expande la representación interna,
- luego aplica operaciones eficientes,
- y después la proyecta nuevamente a una representación más compacta.

Esto se combina con conexiones residuales cuando las dimensiones lo permiten.

### Idea general
MobileNetV2 busca un buen equilibrio entre:

- eficiencia computacional,
- tamaño del modelo,
- y capacidad de representación.

## ¿Qué significa usar MobileNetV2 preentrenado?

Cuando cargamos MobileNetV2 con pesos preentrenados en ImageNet, estamos usando un modelo que ya aprendió patrones visuales generales a partir de una enorme colección de imágenes.

Eso significa que el modelo ya sabe detectar, en distintos niveles:

- bordes,
- texturas,
- formas,
- patrones visuales más complejos.

La idea de **transfer learning** es reutilizar esas representaciones y adaptarlas a nuestra tarea actual: distinguir gatos y perros.

## Preprocesamiento específico del modelo

Un detalle importante en transfer learning es que cada familia de modelos suele esperar un cierto preprocesamiento de entrada.

Para **MobileNetV2**, usaremos la función:

`tf.keras.applications.mobilenet_v2.preprocess_input`

Esto transforma las imágenes al rango y formato esperado por la red preentrenada.

In [14]:
preprocess_input = tf.keras.applications.mobilenet_v2.preprocess_input

## Cargar la base convolucional preentrenada

Usaremos:

- `weights="imagenet"` para cargar pesos preentrenados,
- `include_top=False` para quitar la cabeza original de clasificación,
- `input_shape=(160, 160, 3)` para adaptar el tamaño de entrada a nuestro notebook.

La idea es quedarnos con la **base convolucional** que extrae características visuales.

In [ ]:
base_model = tf.keras.applications.MobileNetV2(
    input_shape=img_size + (3,),
    include_top=False,
    weights="imagenet"
)

base_model.summary()

## Batch Normalization: una idea importante en redes profundas

Al revisar la arquitectura de **MobileNetV2**, notarás que aparecen muchas capas de tipo **BatchNormalization**.

La idea general de Batch Normalization es ayudar a que el entrenamiento de redes profundas sea más estable.

De forma intuitiva, esta capa toma las activaciones de una capa intermedia y las **normaliza por mini-batch**, para que tengan una escala más controlada durante el entrenamiento.

### ¿Por qué puede ser útil?
En redes profundas, los valores intermedios pueden cambiar mucho entre capas y entre iteraciones.  
Eso puede dificultar el entrenamiento.

Batch Normalization ayuda a:

- estabilizar el entrenamiento,
- permitir tasas de aprendizaje razonables,
- acelerar la convergencia en muchos casos,
- y actuar, en cierta medida, como una forma suave de regularización.

## ¿Cómo funciona Batch Normalization?

De manera simplificada, durante entrenamiento la capa:

1. toma las activaciones de un mini-batch,
2. calcula su media,
3. calcula su desviación estándar,
4. normaliza esos valores,
5. y luego aplica una transformación aprendible.

Es decir, no solo normaliza, sino que además aprende dos parámetros:

- un factor de escala,
- y un desplazamiento.

Eso permite que la red conserve flexibilidad.

### Idea importante
Batch Normalization se comporta de manera distinta en:

- **entrenamiento**: usa estadísticas del mini-batch,
- **inferencia**: usa estadísticas acumuladas durante el entrenamiento.

Por eso, en transfer learning suele ser importante tener cuidado con cómo se usa el modelo base cuando contiene capas BatchNormalization.

In [ ]:
bn_layers = [layer for layer in base_model.layers if isinstance(layer, layers.BatchNormalization)]

print("Número de capas BatchNormalization en MobileNetV2:", len(bn_layers))
print("\nPrimeras 10 capas BatchNormalization:")
for layer in bn_layers[:10]:
    print(layer.name)

## ¿Batch Normalization se puede usar en cualquier red neuronal?

**En muchos casos, sí**, aunque depende del tipo de arquitectura y del problema.

Batch Normalization se ha usado ampliamente en:

- redes densas (MLP),
- redes convolucionales (CNN),
- y muchas arquitecturas profundas modernas.

Sin embargo, no siempre es la mejor opción en cualquier contexto.

### Observaciones prácticas
- En **CNN**, Batch Normalization es muy común.
- En **MLP**, también puede usarse, especialmente en redes profundas.
- En modelos secuenciales, como **RNN/LSTM/GRU**, su uso es menos directo y muchas veces se prefieren otras técnicas.
- En arquitecturas más recientes, como transformers, suelen aparecer otras formas de normalización, por ejemplo **Layer Normalization**.

### Mensaje práctico
Batch Normalization es una herramienta muy importante, pero no es una regla obligatoria para toda red neuronal.
Su conveniencia depende de la arquitectura y del tipo de datos.

## ¿Qué significa `include_top=False`?

La parte final original del modelo preentrenado estaba diseñada para clasificar las clases de ImageNet.

Pero nuestra tarea no es esa.

Nosotros no queremos la cabeza original del modelo, sino su capacidad para extraer representaciones visuales útiles.

Por eso usamos:

- `include_top=False`

y luego construiremos nuestra propia cabeza de clasificación para:

- gato,
- perro.

## Primera estrategia: feature extraction

Comenzaremos con la estrategia más simple y habitual:

- cargar el modelo preentrenado,
- **congelar** sus pesos,
- agregar una nueva cabeza,
- entrenar solo esa parte nueva.

Eso significa que MobileNetV2 actuará como un extractor de características ya aprendido.

In [ ]:
base_model.trainable = False
print("Base model trainable:", base_model.trainable)

## Construcción del modelo completo con Functional API

Aquí aparece naturalmente la **Functional API**.

Nuestro flujo será:

1. entrada,
2. data augmentation,
3. preprocesamiento,
4. modelo base preentrenado,
5. global average pooling,
6. dropout,
7. capa final de clasificación.

Esto ya no se siente tanto como una simple pila lineal creada con `Sequential`, sino más bien como una conexión explícita de bloques.

In [18]:
inputs = keras.Input(shape=img_size + (3,))
x = data_augmentation(inputs)
x = preprocess_input(x)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)

model = keras.Model(inputs, outputs)

## Resumen del modelo

Veamos cómo quedó armado el modelo final.

In [ ]:
model.summary()

## Compilación del modelo

Como estamos en una tarea binaria, usaremos:

- `binary_crossentropy` como función de pérdida,
- `accuracy` como métrica.

Partiremos con `Adam`, que ya hemos usado en sesiones anteriores.

In [20]:
model.compile(
    optimizer=keras.optimizers.Adam(),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

## Evaluación inicial antes de entrenar

Antes de entrenar, evaluaremos el modelo en validación.

Esto sirve para recordar que la nueva cabeza todavía no ha aprendido a resolver nuestra tarea.

In [ ]:
initial_loss, initial_accuracy = model.evaluate(val_ds)
print("Initial loss:", initial_loss)
print("Initial accuracy:", initial_accuracy)

## Entrenamiento con feature extraction

Entrenaremos primero solo la cabeza final.

Usaremos pocas épocas para que el notebook sea liviano y para observar rápidamente el comportamiento del modelo.

In [ ]:
initial_epochs = 5

history = model.fit(
    train_ds,
    epochs=initial_epochs,
    validation_data=val_ds
)

## Visualización de curvas de entrenamiento

Graficaremos:

- pérdida de entrenamiento y validación,
- accuracy de entrenamiento y validación.

Esto nos ayuda a detectar si el modelo está aprendiendo y si aparece sobreajuste.

In [ ]:
acc = history.history["accuracy"]
val_acc = history.history["val_accuracy"]

loss = history.history["loss"]
val_loss = history.history["val_loss"]

epochs_range = range(initial_epochs)

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label="Train Accuracy")
plt.plot(epochs_range, val_acc, label="Val Accuracy")
plt.legend(loc="lower right")
plt.title("Training and Validation Accuracy")

plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label="Train Loss")
plt.plot(epochs_range, val_loss, label="Val Loss")
plt.legend(loc="upper right")
plt.title("Training and Validation Loss")

plt.show()

## Nota sobre las curvas: ¿por qué a veces la validación se ve mejor que el entrenamiento?

En este notebook puede ocurrir que la **accuracy de validación** sea ligeramente mayor que la de entrenamiento, o que la **loss de validación** sea menor.

Esto no necesariamente es un problema.

### Una razón importante
Durante entrenamiento aplicamos **data augmentation**, por lo que el modelo ve imágenes más difíciles o más variadas:

- flips,
- rotaciones,
- zooms,
- otras pequeñas transformaciones.

En cambio, en validación el modelo ve imágenes originales, sin augmentation.

### Además, estamos usando una red preentrenada
Como la base de **MobileNetV2** ya aprendió representaciones visuales útiles, puede generalizar bastante bien desde el comienzo sobre imágenes “limpias” de validación.

Por eso puede ocurrir que:

- **train** se vea un poco más difícil,
- mientras **validation** muestre un desempeño algo mejor.

### Idea práctica
Cuando usamos **data augmentation** y **transfer learning**, no siempre debemos esperar que entrenamiento supere a validación en todas las épocas.

Lo importante es mirar el comportamiento general de las curvas y confirmar el desempeño final en el conjunto de **test**.

## Evaluación en test

Ahora evaluamos el modelo en un conjunto no usado ni para entrenar ni para validar.

In [ ]:
test_loss, test_accuracy = model.evaluate(test_ds)
print("Test loss:", test_loss)
print("Test accuracy:", test_accuracy)

## Predicciones sobre algunas imágenes

Además de métricas agregadas, conviene mirar ejemplos concretos.

Esto nos ayuda a interpretar:

- cuándo el modelo acierta,
- cuándo se equivoca,
- y con qué confianza parece estar prediciendo.

In [ ]:
image_batch, label_batch = next(iter(test_ds))
predictions = model.predict(image_batch)
predicted_labels = (predictions > 0.5).astype("int32")

In [ ]:
plt.figure(figsize=(12, 12))

for i in range(9):
    ax = plt.subplot(3, 3, i + 1)
    plt.imshow(image_batch[i].numpy().astype("uint8"))

    true_label = class_names[int(label_batch[i].numpy()[0])]
    pred_label = class_names[int(predicted_labels[i][0])]
    prob = predictions[i][0]

    plt.title(f"Real: {true_label}\nPred: {pred_label} ({prob:.2f})")
    plt.axis("off")

plt.show()

## Segunda etapa: fine-tuning

Ahora haremos un ajuste fino del modelo.

La idea no es entrenar toda la red desde cero, sino:

- descongelar solo una parte final del modelo base,
- usar un learning rate pequeño,
- y continuar el entrenamiento suavemente.

Esto permite adaptar mejor las representaciones al problema específico.

In [28]:
base_model.trainable = True

## ¿Descongelamos todo?

Podríamos hacerlo, pero para un primer ejemplo eso no suele ser lo mejor.

Una estrategia frecuente es mantener congelada gran parte de la base y descongelar solo las capas más altas, que son las más específicas.

Veamos cuántas capas tiene MobileNetV2.

In [ ]:
print("Number of layers in the base model:", len(base_model.layers))

## Descongelar solo las últimas capas

Aquí elegiremos un punto de corte.

Las capas anteriores quedarán congeladas y las capas posteriores quedarán entrenables.

Este número no es universal: depende del modelo, del tamaño del dataset y del problema.

In [30]:
fine_tune_at = 100

for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

## Recompilar el modelo

Después de cambiar qué capas son entrenables, es importante recompilar.

Además, en fine-tuning usamos un **learning rate más pequeño** para no modificar demasiado rápido los pesos preentrenados.

In [31]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

## Continuar entrenamiento con fine-tuning

Entrenaremos unas pocas épocas adicionales.

In [ ]:
fine_tune_epochs = 5
total_epochs = initial_epochs + fine_tune_epochs

history_fine = model.fit(
    train_ds,
    epochs=total_epochs,
    initial_epoch=history.epoch[-1] + 1,
    validation_data=val_ds
)

## Combinar historiales

Para comparar mejor, uniremos las curvas de la primera etapa y la etapa de fine-tuning.

In [33]:
acc += history_fine.history["accuracy"]
val_acc += history_fine.history["val_accuracy"]

loss += history_fine.history["loss"]
val_loss += history_fine.history["val_loss"]

In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(acc, label="Train Accuracy")
plt.plot(val_acc, label="Val Accuracy")
plt.axvline(x=initial_epochs - 1, linestyle="--", label="Start Fine-Tuning")
plt.legend(loc="lower right")
plt.title("Training and Validation Accuracy")

plt.subplot(1, 2, 2)
plt.plot(loss, label="Train Loss")
plt.plot(val_loss, label="Val Loss")
plt.axvline(x=initial_epochs - 1, linestyle="--", label="Start Fine-Tuning")
plt.legend(loc="upper right")
plt.title("Training and Validation Loss")

plt.show()

## Evaluación final en test

Ahora medimos nuevamente el desempeño después del fine-tuning.

In [ ]:
test_loss_ft, test_accuracy_ft = model.evaluate(test_ds)
print("Test loss after fine-tuning:", test_loss_ft)
print("Test accuracy after fine-tuning:", test_accuracy_ft)

## Reflexión final

En este notebook vimos dos etapas:

### 1. Feature extraction
- usamos MobileNetV2 preentrenado,
- congelamos la base,
- y entrenamos solo una nueva cabeza de clasificación.

### 2. Fine-tuning
- descongelamos parte de la base,
- usamos una tasa de aprendizaje pequeña,
- y adaptamos el modelo con más cuidado a nuestra tarea.

### Idea central
Transfer learning permite aprovechar conocimiento ya aprendido por una red profunda y reutilizarlo en un nuevo problema, especialmente cuando no tenemos un dataset enorme.

## Preguntas para pensar

1. ¿Por qué conviene congelar la base al comienzo?
2. ¿Por qué en fine-tuning usamos un learning rate pequeño?
3. ¿Qué ventaja práctica tiene usar un modelo preentrenado en vez de entrenar una CNN profunda desde cero?
4. ¿Qué podría pasar si olvidamos aplicar el preprocesamiento correcto para MobileNetV2?

## Bonus: usar la base convolucional como extractor de features

Hasta ahora usamos la base preentrenada como parte de un modelo completo en Keras.

Pero también podemos usarla de otra forma:

1. pasar cada imagen por la base convolucional,
2. extraer un vector de características,
3. y luego entrenar un modelo clásico de machine learning sobre esos vectores.

En este ejemplo usaremos un **Random Forest**.

### Idea práctica
La CNN preentrenada actúa aquí como un **extractor de representaciones visuales**.

## Preparar un extractor de features

Usaremos la misma base convolucional preentrenada, pero ahora nos interesa obtener un vector fijo por imagen.

Para eso construiremos un modelo auxiliar que haga:

- preprocesamiento,
- paso por la base convolucional,
- global average pooling.

La salida será un vector de features por imagen.

In [ ]:
feature_extractor = keras.Sequential([
    layers.Input(shape=img_size + (3,)),
    layers.Lambda(preprocess_input),
    base_model,
    layers.GlobalAveragePooling2D()
])

feature_extractor.summary()

`layers.Lambda(...)` permite incorporar una función dentro del modelo como si fuera una capa.
En este caso se usa para aplicar `preprocess_input` antes de pasar la imagen por la red preentrenada.
No aprende parámetros; solo transforma la entrada al formato esperado por el modelo.

## Función para extraer features y etiquetas

Ahora recorreremos un dataset por batches y construiremos:

- una matriz `X` con features,
- un vector `y` con etiquetas.

In [37]:
def extract_features(dataset, extractor):
    features_list = []
    labels_list = []

    for images, labels in dataset:
        features = extractor.predict(images, verbose=0)
        features_list.append(features)
        labels_list.append(labels.numpy().ravel())

    X = np.concatenate(features_list, axis=0)
    y = np.concatenate(labels_list, axis=0)

    return X, y

## Extraer features para train y test

Aquí usaremos la base preentrenada congelada para generar representaciones de las imágenes.

In [ ]:
X_train_feat, y_train_feat = extract_features(train_ds, feature_extractor)
X_test_feat, y_test_feat = extract_features(test_ds, feature_extractor)

print("Shape X_train_feat:", X_train_feat.shape)
print("Shape y_train_feat:", y_train_feat.shape)
print("Shape X_test_feat:", X_test_feat.shape)
print("Shape y_test_feat:", y_test_feat.shape)

## Entrenar un Random Forest sobre los features

Ahora usaremos estos vectores como entrada para un modelo clásico de machine learning.

In [39]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train_feat, y_train_feat)

## Evaluación del Random Forest

In [ ]:
y_pred_rf = rf_model.predict(X_test_feat)

rf_acc = accuracy_score(y_test_feat, y_pred_rf)
print("Random Forest test accuracy:", rf_acc)

In [ ]:
print(classification_report(y_test_feat, y_pred_rf, target_names=class_names))

In [ ]:
import seaborn as sns

cm = confusion_matrix(y_test_feat, y_pred_rf)

plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix - Random Forest on Deep Features")
plt.show()

## Interpretación

Este resultado muestra que una red preentrenada puede servir no solo para construir modelos de deep learning end-to-end.

También puede usarse como un generador de features de alta calidad, sobre los cuales luego entrenamos un modelo clásico.

### Idea central
La base convolucional preentrenada transforma imágenes en representaciones útiles.
Luego, otro modelo puede aprender a clasificar a partir de esas representaciones.

## Comparación conceptual

Aquí vimos dos estrategias diferentes:

### 1. Modelo completo en Keras
- entrada: imagen
- salida: clase
- entrenamiento: cabeza nueva y luego fine-tuning

### 2. CNN preentrenada + Random Forest
- entrada: imagen
- salida intermedia: vector de features
- entrenamiento final: modelo clásico de machine learning

### Pregunta para pensar
¿Cuáles podrían ser las ventajas y desventajas de cada enfoque?

## Bonus: modelos preentrenados disponibles en Keras

En este notebook usamos **MobileNetV2**, pero Keras incluye varias familias de modelos preentrenados listas para usar.

Algunos ejemplos conocidos son:

- `MobileNetV2`
- `ResNet50`
- `EfficientNetB0`
- `DenseNet121`
- `Xception`
- `InceptionV3`
- `VGG16`
- `VGG19`

La idea general de uso es muy parecida:

1. importar la arquitectura,
2. cargar pesos preentrenados,
3. decidir si usamos o no la cabeza original,
4. agregar nuestra propia cabeza para la nueva tarea.

In [ ]:
from tensorflow.keras.applications import (
    MobileNetV2,
    ResNet50,
    EfficientNetB0,
    DenseNet121,
    Xception,
    InceptionV3,
    VGG16,
    VGG19
)

print("Arquitecturas importadas correctamente.")

## Ejemplo: cómo cargar distintas arquitecturas preentrenadas

La forma más habitual de cargar una red preentrenada en Keras es usando parámetros como:

- `weights="imagenet"` para cargar pesos preentrenados,
- `include_top=False` para quitar la cabeza original,
- `input_shape=...` para definir el tamaño de entrada.

A continuación mostramos algunos ejemplos.

In [ ]:
mobilenet_base = MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(160, 160, 3)
)

resnet_base = ResNet50(
    weights="imagenet",
    include_top=False,
    input_shape=(160, 160, 3)
)

efficientnet_base = EfficientNetB0(
    weights="imagenet",
    include_top=False,
    input_shape=(160, 160, 3)
)

print("MobileNetV2 output shape:", mobilenet_base.output_shape)
print("ResNet50 output shape:", resnet_base.output_shape)
print("EfficientNetB0 output shape:", efficientnet_base.output_shape)

## Observación importante

Aunque la forma de carga es parecida, **no todos los modelos esperan exactamente el mismo preprocesamiento**.

Por ejemplo, muchas familias tienen su propia función:

- `mobilenet_v2.preprocess_input`
- `resnet.preprocess_input`
- `xception.preprocess_input`
- etc.

Además, algunas arquitecturas requieren tamaños de entrada más típicos que otras.

### Mensaje práctico
La lógica de transfer learning es muy parecida entre modelos, pero siempre conviene revisar:

- el tamaño de entrada,
- el preprocesamiento esperado,
- el costo computacional,
- y el número de parámetros.

In [ ]:
print("Número de parámetros en MobileNetV2:", mobilenet_base.count_params())
print("Número de parámetros en ResNet50:", resnet_base.count_params())
print("Número de parámetros en EfficientNetB0:", efficientnet_base.count_params())

## Idea final del bonus

Una vez entendido el flujo de transfer learning, cambiar de arquitectura suele ser bastante simple.

Muchas veces basta con reemplazar la línea donde cargamos el modelo base y ajustar:

- el preprocesamiento,
- el tamaño de entrada,
- y eventualmente algunos hiperparámetros.

In [ ]:
# Ejemplo: cambiar solo la arquitectura base

base_model = tf.keras.applications.ResNet50(
    input_shape=img_size + (3,),
    include_top=False,
    weights="imagenet"
)

base_model.trainable = False

print("Nueva base cargada:", base_model.name)

Referencia oficial:
Keras Applications (documentación oficial de Keras)

https://keras.io/api/applications/

## Bonus: ¿Existe transfer learning para datos secuenciales?

Sí, **también existe transfer learning para datos secuenciales**.

La idea general es la misma que en imágenes:

- partir desde un modelo que ya aprendió representaciones útiles,
- y reutilizar ese conocimiento en una nueva tarea.

Sin embargo, en secuencias el panorama es un poco distinto al de visión computacional.

En imágenes, es muy común reutilizar directamente una CNN preentrenada sobre ImageNet.

En cambio, en secuencias, el transfer learning suele aparecer más frecuentemente a través de:

- **embeddings preentrenados**,
- **encoders preentrenados**,
- y, en años recientes, sobre todo mediante **transformers preentrenados**.

## ¿Y qué pasa con RNN, LSTM y GRU?

Conceptualmente, **sí se puede** aplicar transfer learning a modelos como:

- RNN,
- LSTM,
- GRU.

Por ejemplo, se podrían reutilizar pesos aprendidos previamente en una tarea relacionada, o reutilizar representaciones ya entrenadas.

Pero en la práctica actual, especialmente en tareas de texto, **no es tan común hablar de una LSTM preentrenada del mismo modo en que hablamos de una ResNet o una MobileNet preentrenada**.

Hoy, cuando se habla de modelos secuenciales preentrenados, lo más frecuente es encontrarse con:

- modelos de lenguaje preentrenados,
- transformers,
- embeddings reutilizables.

## Una comparación útil

### En visión computacional
Es muy común usar modelos como:

- MobileNet
- ResNet
- EfficientNet

preentrenados sobre grandes datasets de imágenes.

### En secuencias y texto
Es común reutilizar:

- embeddings preentrenados,
- modelos de lenguaje preentrenados,
- encoders basados en transformers.

### Idea clave
Sí existe transfer learning en secuencias, pero el ecosistema moderno está mucho más dominado por **transformers** que por **RNN/LSTM/GRU** clásicas.

## Entonces, ¿hay un equivalente exacto a MobileNetV2 pero para LSTM?

No exactamente en el mismo sentido.

La analogía correcta sería esta:

- en imágenes, reutilizamos una CNN preentrenada;
- en texto moderno, muchas veces reutilizamos un **transformer preentrenado**;
- en tareas más clásicas, también podemos reutilizar **embeddings** o componentes entrenados previamente.

Por eso, si uno piensa en transfer learning en secuencias, la respuesta corta es:

**sí existe, pero hoy suele aparecer más en transformers que en redes recurrentes clásicas.**

## Conexión con la próxima sesión

Esto nos deja una transición natural hacia la próxima clase.

En la sesión anterior vimos:

- RNN,
- LSTM,
- GRU.

En esta sesión vimos transfer learning en visión computacional.

Y en la próxima sesión veremos que, para muchas tareas secuenciales modernas, el paradigma dominante de modelos preentrenados pasa por los **transformers**.

## Mensaje final del bonus

La idea de **reutilizar conocimiento aprendido** no es exclusiva de las CNN ni de las imágenes.

Es una idea más general del deep learning moderno.

Lo que cambia entre dominios no es tanto la idea de transfer learning, sino **qué tipo de arquitectura y qué tipo de representación preentrenada se reutiliza**.